In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

/data/hyeryung/.conda/envs/loc-edit/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
import pandas as pd
import numpy as np

In [3]:
model1=AutoModelForSequenceClassification.from_pretrained('/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint')
tokenizer1=AutoTokenizer.from_pretrained('/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint')
model_cls1=AutoModelForSequenceClassification.from_pretrained('/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier/step_500_best_checkpoint/')
model2=AutoModelForSequenceClassification.from_pretrained('/data/hyeryung/loc_edit/models/roberta-base-yelp-sentiment-classifier-energy-training/step_81900_best_checkpoint')
model_cls2=AutoModelForSequenceClassification.from_pretrained('/data/hyeryung/loc_edit/models/roberta-base-yelp-sentiment-classifier/step_83000')
tokenizer2=AutoTokenizer.from_pretrained('/data/hyeryung/loc_edit/models/roberta-base-yelp-sentiment-classifier-energy-training/step_81900_best_checkpoint')
validset1=pd.read_json('data/toxicity/jigsaw-unintended-bias-in-toxicity-classification/fine-grained/valid.jsonl', lines=True)
validset2=pd.read_json('data/yelp/valid.jsonl', lines=True)
testset1=pd.read_json('data/toxicity/jigsaw-unintended-bias-in-toxicity-classification/fine-grained/test.jsonl', lines=True)
testset2=pd.read_json('data/yelp/test.jsonl', lines=True)

In [ ]:
def evaluate_model(model, tokenizer, dataset, batch_size=16):
    """
    Evaluates a model on a given dataset and returns accuracy and RMSE.
    """
    model.eval()
    texts = dataset['text'].tolist()
    labels = dataset['label'].tolist()
    
    preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_encodings = tokenizer(batch_texts, truncation=True, padding=True, return_tensors='pt')
            outputs = model(**batch_encodings).logits
            
            batch_preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds.extend(batch_preds)
    
    accuracy = accuracy_score(labels, preds)
    rmse = np.sqrt(mean_squared_error(labels, preds))
    return accuracy, rmse

# Evaluate both models
accuracy1, rmse1 = evaluate_model(model1, tokenizer1, validset1)
accuracy2, rmse2 = evaluate_model(model2, tokenizer2, validset2)

print(f'Toxicity Model - Accuracy: {accuracy1:.4f}, RMSE: {rmse1:.4f}')
print(f'Yelp Sentiment Model - Accuracy: {accuracy2:.4f}, RMSE: {rmse2:.4f}')


In [ ]:
# validset
## classification accuracy 



## rmse
